In [194]:
import pandas as pd
import numpy as np
import os
import math
from datetime import datetime

pd.set_option('display.max_colwidth', None)

In [158]:
# Load dfs
directory = r'data/monthly_costs/'
columns = ['date', 'name', 'fin_date', 'value', 'account', 'nan', 'code']
dfs = []

for filename in os.listdir(directory):
    if filename.endswith(".txt"):
        filepath = os.path.join(directory, filename)
        df = pd.read_csv(filepath, sep='|', header=None, names=columns, encoding='latin1')
        dfs.append(df)
df = pd.concat(dfs)

df['date'] = pd.to_datetime(df['date'], dayfirst=True)
# df['consecutive'] = df['date'].dt.year + (df['date'].dt.month /12)
df = df.drop(['nan', 'fin_date', 'account'], axis=1)
df = df.drop_duplicates()


In [159]:
# filter into excluded_df and included_df
include = [
    "SUPER VINI",
    "ALCAMPO",
    "MARKET",
    "FRUTA",
    "mercadona",
    "carniceria",
    "veterinaria",
    "carref",
    "hiper",
    "papa jonhs",
    "polleria",
    "organic shop",
    "LIDL",
    "Simyo",
    "alejandro",
    "SANTIAGO",
    "Amon",
    "SEGUROS ADESLAS",
    "ELECTRICIDAD IBERDROLA",
    "AGUA CANAL",
    "LAS ROZAS",
    "ELECTRICIDAD FACTOR ENERGIA",
    "PESCADERIA",
    "dulce",
    "quesos",
    "paladar",
    "IBER LIMOSTAR"
    ]
include = [s.lower().strip() for s in include]

exclude = [
    "IKEA",
    "EL CORTE INGLES",
    "LEROY MERLIN",
    "AUTOESCUELA",
    "XE EUROPE",
    "Decorabano-Armilla",
    "Arciniega",
    "TRASPASO",
    "IBERIA",
    "COMISIONES",
    "INTERESES",
    "RECARGA",
    "IBERDROLA OTROS 2"
    ]
exclude = [s.lower().strip() for s in exclude]

def filter_rows(df):
    df_in  = df[df['name'].str.contains('|'.join(include), case=False, na=False)]
    df_in = df_in[~df_in['name'].str.contains('|'.join(exclude), case=False, na=False)]

    df_ex  = df[~df['name'].str.contains('|'.join(include), case=False, na=False)]
    df_ex = df_ex[df_ex['name'].str.contains('|'.join(exclude), case=False, na=False)]

    return df_in, df_ex

included_df, excluded_df = filter_rows(df)

In [233]:
# Calculate months share
def filter_by_date(df, month, year):

    return df[(df['date'].dt.month == month) & (df['date'].dt.year == year)]

def calculate_share(df, month, year, cash, remainder):
    costs = filter_by_date(df, month, year)
    costs = costs[costs['value'] < 0]
    total_costs = costs.value.sum() - cash + remainder

    return -(total_costs/4)
    
def get_previous_date(month, year):
    month -= 1
    if month == 0:
        month = 12
        year = year - 1

    return month, year


year = datetime.now().year
month, year = get_previous_date(datetime.now().month, year)
df = pd.read_csv(r'data/monthly_costs.csv')
last_months_cash = df.loc[df['month'] == month].cash.iloc[0]
remainder = 0.16
my_share = round(calculate_share(included_df, month, year, last_months_cash, remainder))
print(my_share)

if ((df['month'] == datetime.now().month) & (df['year'] == datetime.now().year)).any() == False:
    pd.concat([df, pd.DataFrame({'cash': my_share, 'month': datetime.now().month, 'year': datetime.now().year}, index=[0])]).to_csv(r'data/monthly_costs.csv', index=False)

260


In [140]:
filter_by_date(included_df[included_df['value'] < 0], 6, 2024)#.value.sum() - cash + remainder

,date,name,value,code
2,2024-06-26,COMPRA TARJ. 5402XXXXXXXX9016 MARKET LAS ROZA-LAS ROZAS,-17.69,5402__9016
6,2024-06-24,COMPRA TARJ. 5402XXXXXXXX9016 MARKET LAS ROZA-LAS ROZAS,-8.08,5402__9016
7,2024-06-21,COMPRA TARJ. 5402XXXXXXXX9016 IBER LIMOSTAR-FUENLABRADA,-42.90,5402__9016
8,2024-06-21,COMPRA TARJ. 5402XXXXXXXX9016 MARKET LAS ROZA-LAS ROZAS,-12.23,5402__9016
11,2024-06-06,COMPRA TARJ. 5402XXXXXXXX9016 EL HIPER DE LA FRUTA-LAS ROZAS DE,-13.60,5402__9016
12,2024-06-04,COMPRA TARJ. 5402XXXXXXXX9016 CARREF MAJADAHONDA-MADRID,-18.06,5402__9016
1,2024-06-25,COMPRA TARJ. 5402XXXXXXXX6027 ESPACIO EZZIANI-LAS ROZAS DE,-3.40,5402__6027
0,2024-06-28,COMPRA TARJ. 5402XXXXXXXX3027 SUPER VINI-ROZAS LAS,-0.99,5402__3027
1,2024-06-28,COMPRA TARJ. 5402XXXXXXXX3027 EL HIPER DE LA FRUTA-LAS ROZAS DE,-7.49,5402__3027
3,2024-06-26,"ELECTRICIDAD IBERDROLA CLIENTES,SA IBERDROLA ELECTRI",-68.99,268456693000
